<a href="https://colab.research.google.com/github/sajancodes/project-xyz/blob/main/notebooks/qwen3_4b_coding_stream_qlora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project XYZ — Track B: Experiment V2 (16K Streaming SFT for Coding & Capability Preservation)

This notebook executes **Experiment V2** on **`Qwen/Qwen3-4B-Instruct-2507`** using **4-bit NF4 QLoRA** on an **NVIDIA T4 GPU (16 GB VRAM)** in Google Colab.

### Key Experiment V2 Upgrades Over V1:
1. **True Progressive Streaming**: Consumes 16,000 unique examples dynamically from Hugging Face with `streaming=True` without downloading full datasets.
2. **Targeted Mixture**:
   - **45% OpenCoder** (`educational_instruct`): ~7,200 educational coding instruction pairs.
   - **25% BigCode** (`self-oss-instruct-sc2-exec-filter-50k`): ~4,000 execution-validated coding tasks.
   - **20% SmolTalk** (`smol-magpie-ultra`): ~3,200 general instruction and reasoning samples.
   - **10% Cybersecurity** (`Trendyol` + curated seeds): ~1,600 unique defensive security dialogues.
3. **Longer Context**: Full `MAX_SEQ_LENGTH = 2048` with assistant-only loss masking.
4. **Refined Hyperparameters**: LoRA rank $r=16, \alpha=32$, learning rate $5 \times 10^{-5}$, `max_steps=1000` (effective batch size 16 = 16,000 example exposures).
5. **Execution-Based Coding Benchmark**: Evaluates code correctness via a sandboxed unit test runner on 60 Python programming tasks.
6. **Google Drive Persistence**: Checkpoints stream to `MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2/`.

In [1]:
# CELL 2: Kill Stale Colab GPU Processes + Full VRAM Cleanup + GPU Verification

import os
import gc
import time
import signal
import subprocess
import torch


print("=" * 90)
print("COLAB GPU PROCESS + VRAM CLEANUP")
print("=" * 90)


# ============================================================
# 1. Verify CUDA
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Connect this notebook to a Colab T4 GPU runtime."
    )

CURRENT_PID = os.getpid()

print(f"Current notebook PID: {CURRENT_PID}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 2. Find all processes currently using GPU memory
# ============================================================

def get_gpu_processes():
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    processes = []

    for line in result.stdout.strip().splitlines():
        if not line.strip():
            continue

        parts = [p.strip() for p in line.split(",")]

        if len(parts) < 3:
            continue

        try:
            pid = int(parts[0])
            process_name = parts[1]
            memory_mb = int(parts[2])

            processes.append(
                {
                    "pid": pid,
                    "name": process_name,
                    "memory_mb": memory_mb,
                }
            )

        except ValueError:
            continue

    return processes


gpu_processes = get_gpu_processes()

print("\nGPU processes before cleanup:")

if not gpu_processes:
    print("  None")
else:
    for proc in gpu_processes:
        marker = " ← CURRENT NOTEBOOK" if proc["pid"] == CURRENT_PID else ""
        print(
            f"  PID {proc['pid']:<8} "
            f"{proc['memory_mb']:>6} MiB  "
            f"{proc['name']}"
            f"{marker}"
        )


# ============================================================
# 3. Kill stale Python GPU processes
# ============================================================

killed = []

for proc in gpu_processes:

    pid = proc["pid"]

    # NEVER kill current notebook kernel
    if pid == CURRENT_PID:
        continue

    process_name = proc["name"].lower()

    # Only target Python processes.
    # Avoid touching unrelated GPU/system processes.
    if "python" not in process_name:
        print(
            f"Skipping non-Python GPU process "
            f"PID {pid}: {proc['name']}"
        )
        continue

    try:
        print(
            f"\nKilling stale GPU Python process "
            f"PID {pid} using {proc['memory_mb']} MiB..."
        )

        os.kill(pid, signal.SIGKILL)

        killed.append(pid)

    except ProcessLookupError:
        print(f"PID {pid} already exited.")

    except PermissionError:
        print(f"Could not kill PID {pid}: permission denied.")

    except Exception as e:
        print(f"Could not kill PID {pid}: {e}")


# Give NVIDIA driver a moment to release VRAM
if killed:
    print("\nWaiting for GPU memory to be released...")
    time.sleep(3)


# ============================================================
# 4. Delete known notebook GPU objects
# ============================================================

objects_to_delete = [
    "model",
    "trainer",
    "optimizer",
    "scheduler",
    "train_output",
    "train_stream",
    "val_dataset",
    "test_dataset",
    "collator",
    "base_results",
    "base_multi_results",
    "base_coding_results",
    "lora_config",
]

deleted_objects = []

for name in objects_to_delete:

    if name in globals():

        try:
            del globals()[name]
            deleted_objects.append(name)
        except Exception:
            pass


if deleted_objects:
    print("\nDeleted notebook objects:")
    for name in deleted_objects:
        print(f"  - {name}")


# ============================================================
# 5. Python + CUDA cleanup
# ============================================================

for _ in range(3):
    gc.collect()

torch.cuda.empty_cache()

try:
    torch.cuda.ipc_collect()
except Exception:
    pass

try:
    torch.cuda.synchronize()
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# 6. Reset PyTorch memory stats
# ============================================================

try:
    torch.cuda.reset_peak_memory_stats()
except Exception:
    pass

try:
    torch.cuda.reset_accumulated_memory_stats()
except Exception:
    pass


# ============================================================
# 7. Report final GPU status
# ============================================================

free_bytes, total_bytes = torch.cuda.mem_get_info()

free_gb = free_bytes / 1024**3
total_gb = total_bytes / 1024**3
used_gb = total_gb - free_gb

allocated_gb = torch.cuda.memory_allocated(0) / 1024**3
reserved_gb = torch.cuda.memory_reserved(0) / 1024**3


print("\n" + "=" * 90)
print("GPU STATUS AFTER CLEANUP")
print("=" * 90)

print(f"GPU:                 {torch.cuda.get_device_name(0)}")
print(f"Current notebook PID:{CURRENT_PID}")
print(f"Total VRAM:          {total_gb:.2f} GB")
print(f"Free VRAM:           {free_gb:.2f} GB")
print(f"Used VRAM:           {used_gb:.2f} GB")
print(f"PyTorch allocated:   {allocated_gb:.2f} GB")
print(f"PyTorch reserved:    {reserved_gb:.2f} GB")


# ============================================================
# 8. Show remaining GPU processes
# ============================================================

remaining = get_gpu_processes()

print("\nRemaining GPU processes:")

if not remaining:
    print("  None")
else:
    for proc in remaining:

        marker = (
            " ← CURRENT NOTEBOOK"
            if proc["pid"] == CURRENT_PID
            else ""
        )

        print(
            f"  PID {proc['pid']:<8} "
            f"{proc['memory_mb']:>6} MiB  "
            f"{proc['name']}"
            f"{marker}"
        )


# ============================================================
# 9. Safety check
# ============================================================

print("\n" + "=" * 90)

if free_gb >= 12:

    print("✅ GPU CLEAN — enough free VRAM to load Qwen3-4B NF4.")

elif free_gb >= 10:

    print(
        "⚠️ GPU mostly clean, but VRAM is somewhat limited.\n"
        "Qwen3-4B NF4 may still fit."
    )

else:

    print(
        "❌ GPU is still holding too much memory.\n"
        "Disconnect the Colab runtime completely and reconnect "
        "to a fresh T4 before loading Qwen."
    )

print("=" * 90)


# ============================================================
# 10. Driver-level report
# ============================================================

print("\nNVIDIA-SMI:\n")

subprocess.run(["nvidia-smi"])

COLAB GPU PROCESS + VRAM CLEANUP
Current notebook PID: 1732
GPU: Tesla T4

GPU processes before cleanup:
  None

GPU STATUS AFTER CLEANUP
GPU:                 Tesla T4
Current notebook PID:1732
Total VRAM:          14.56 GB
Free VRAM:           14.46 GB
Used VRAM:           0.10 GB
PyTorch allocated:   0.00 GB
PyTorch reserved:    0.00 GB

Remaining GPU processes:
  PID 1732        102 MiB  /usr/bin/python3 ← CURRENT NOTEBOOK

✅ GPU CLEAN — enough free VRAM to load Qwen3-4B NF4.

NVIDIA-SMI:



CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
# CELL 3: Mount and Verify Google Drive (Experiment V2 Directory)
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"Google Drive mount skipped (or running locally): {e}")
    DRIVE_MOUNTED = False

DRIVE_ROOT = "/content/drive/MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2"
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/adapters", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/benchmarks", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/reports", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)

print(f"Experiment V2 Google Drive root verified at: {DRIVE_ROOT}")

Google Drive mount skipped (or running locally): mount failed
Experiment V2 Google Drive root verified at: /content/drive/MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2


In [3]:
# CELL 4: Clone / Access Repository on Branch 'qwen3-4b-stream-coding-v2'
import os, sys
from pathlib import Path

REPO_NAME = "project-xyz"
LOCAL_REPO = Path(f"/content/{REPO_NAME}")

if not LOCAL_REPO.exists() and not Path("src").exists():
    print("Cloning repository branch qwen3-4b-stream-coding-v2 into Colab runtime...")
    !git clone -b qwen3-4b-stream-coding-v2 https://github.com/sajancodes/project-xyz.git /content/project-xyz

if LOCAL_REPO.exists():
    os.chdir(str(LOCAL_REPO))

repo_path = str(Path.cwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Active working directory:", os.getcwd())
assert Path("src").exists(), "src directory not found!"

Cloning repository branch qwen3-4b-stream-coding-v2 into Colab runtime...
Cloning into '/content/project-xyz'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 327 (delta 100), reused 284 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (327/327), 590.31 KiB | 2.32 MiB/s, done.
Resolving deltas: 100% (100/100), done.
Active working directory: /content/project-xyz


In [4]:
# CELL 5: Install Pinned QLoRA Dependencies
!pip install --upgrade pip
!pip install torch>=2.2.0 transformers>=4.48.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 peft>=0.14.0 trl>=0.12.0 datasets>=2.21.0 pyyaml scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [5]:
# CELL 6: Print Dependency Versions
import sys, torch, transformers, peft, trl, accelerate, bitsandbytes

print("=" * 50)
print("INSTALLED FRAMEWORK VERSIONS")
print("=" * 50)
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"Accelerate:   {accelerate.__version__}")
print(f"BitsAndBytes: {bitsandbytes.__version__}")
print("=" * 50)

INSTALLED FRAMEWORK VERSIONS
Python:       3.13.15
PyTorch:      2.11.0+cu130 (CUDA: 13.0)
Transformers: 5.20.1
PEFT:         0.21.2
TRL:          1.16.0
Accelerate:   1.15.0
BitsAndBytes: 0.50.2


In [6]:
# CELL 7: Experiment V2 Hyperparameters — 5K-Step Streaming Run

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# ============================================================
# Context / Memory
# ============================================================

# 2048 caused CUDA OOM on the T4.
# Start with 1024 and verify it using the production-length
# forward/backward smoke test in Cell 13.
MAX_SEQ_LENGTH = 1024


# ============================================================
# QLoRA
# ============================================================

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# ============================================================
# Optimization
# ============================================================

LEARNING_RATE = 5e-5

MICRO_BATCH_SIZE = 1

GRADIENT_ACCUMULATION = 16

# Effective batch:
# 1 × 16 = 16 examples / optimizer step
EFFECTIVE_BATCH_SIZE = (
    MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION
)


# ============================================================
# Training Length
# ============================================================

MAX_STEPS = 5000

# Approximate total example exposures:
# 5000 × 16 = 80,000
TARGET_EXAMPLE_EXPOSURES = (
    MAX_STEPS * EFFECTIVE_BATCH_SIZE
)


# ============================================================
# Scheduler
# ============================================================

# Cell 15 passes this as:
# warmup_steps=WARMUP_RATIO
#
# With the currently installed Transformers API,
# 0.05 represents approximately 5% warmup.
WARMUP_RATIO = 0.05


# ============================================================
# Resume / Persistence
# ============================================================

RESUME_FROM_CHECKPOINT = None

DRIVE_ROOT = (
    "/content/drive/MyDrive/ProjectXYZ/"
    "qwen3-4b-coding-stream-v2"
)


# ============================================================
# Summary
# ============================================================

print("=" * 80)
print("EXPERIMENT V2 — 5K STREAMING QLORA")
print("=" * 80)

print(f"Base model:               {BASE_MODEL}")
print(f"Max sequence length:      {MAX_SEQ_LENGTH}")
print(f"LoRA rank:                {LORA_R}")
print(f"LoRA alpha:               {LORA_ALPHA}")
print(f"LoRA dropout:             {LORA_DROPOUT}")

print(f"Learning rate:            {LEARNING_RATE}")
print(f"Micro batch size:         {MICRO_BATCH_SIZE}")
print(f"Gradient accumulation:    {GRADIENT_ACCUMULATION}")
print(f"Effective batch size:     {EFFECTIVE_BATCH_SIZE}")

print(f"Maximum optimizer steps:  {MAX_STEPS:,}")
print(
    f"Approx example exposures: "
    f"{TARGET_EXAMPLE_EXPOSURES:,}"
)

print(
    f"Warmup:                   "
    f"{WARMUP_RATIO * 100:.1f}%"
)

print(f"Drive root:               {DRIVE_ROOT}")

print("=" * 80)
print(
    "Configuration ready. "
    "Cell 13 must pass at MAX_SEQ_LENGTH "
    "before starting full training."
)

EXPERIMENT V2 — 5K STREAMING QLORA
Base model:               Qwen/Qwen3-4B-Instruct-2507
Max sequence length:      1024
LoRA rank:                16
LoRA alpha:               32
LoRA dropout:             0.05
Learning rate:            5e-05
Micro batch size:         1
Gradient accumulation:    16
Effective batch size:     16
Maximum optimizer steps:  5,000
Approx example exposures: 80,000
Warmup:                   5.0%
Drive root:               /content/drive/MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2
Configuration ready. Cell 13 must pass at MAX_SEQ_LENGTH before starting full training.


In [7]:
# CELL 8: Materialize Finite Validation and Test Datasets from Streams
from src.streaming.sampler import build_finite_eval_datasets

print("Building finite validation (600) and test (400) datasets from streams...")
val_path, test_path = build_finite_eval_datasets(
    output_dir="data/qwen4b/processed",
    val_target=600,
    test_target=400,
)
print(f"Finite evaluation datasets ready:")
print(f"  - Validation: {val_path}")
print(f"  - Test:       {test_path}")

Building finite validation (600) and test (400) datasets from streams...


README.md:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

Finite evaluation datasets ready:
  - Validation: data/qwen4b/processed/stream_val.jsonl
  - Test:       data/qwen4b/processed/stream_test.jsonl


In [ ]:
# CELL 9: Load Qwen Tokenizer
from src.models.qwen4b.loader import load_qwen_tokenizer

tokenizer = load_qwen_tokenizer(BASE_MODEL)
print(f"Official Qwen tokenizer loaded. Vocab size: {len(tokenizer):,}")
print(f"Pad token: '{tokenizer.pad_token}', EOS token: '{tokenizer.eos_token}'")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Official Qwen tokenizer loaded. Vocab size: 151,669
Pad token: '<|endoftext|>', EOS token: '<|im_end|>'


In [ ]:
# CELL 10: Load Qwen 4B in NF4 4-bit — T4 GPU ONLY

import gc
import torch

from src.models.qwen4b.loader import load_qwen_model


# ============================================================
# Verify CUDA
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Connect the notebook to a Colab T4 GPU."
    )

print("=" * 80)
print("QWEN3 4B NF4 MODEL LOAD")
print("=" * 80)

print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# Clear stale CUDA allocations
# ============================================================

# Remove old objects if this cell is being rerun
for name in [
    "trainer",
    "model",
    "train_output",
    "train_stream",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()

try:
    torch.cuda.ipc_collect()
except Exception:
    pass


free_bytes, total_bytes = torch.cuda.mem_get_info()

print(
    f"GPU free before model load: "
    f"{free_bytes / 1024**3:.2f} GB / "
    f"{total_bytes / 1024**3:.2f} GB"
)

# A clean T4 should have most of its ~14.5 GiB available here.
if free_bytes / 1024**3 < 10:
    print(
        "\n⚠️ WARNING: Less than 10 GB VRAM is free before loading."
        "\nIf this is immediately after an OOM, restart the Colab kernel."
    )


# ============================================================
# Load the ENTIRE quantized model on GPU 0
# ============================================================

print(
    f"\nLoading {BASE_MODEL} in 4-bit NF4 "
    "(FP16 compute) onto Tesla T4...\n"
)

model = load_qwen_model(
    model_id_or_path=BASE_MODEL,
    load_in_4bit=True,
    compute_dtype="float16",

    # IMPORTANT:
    # Do NOT use "auto" here.
    # Force all quantized model layers onto GPU 0.
    device_map={"": 0},

    trust_remote_code=True,
    use_cache=False,
    enable_gradient_checkpointing=True,
)


# ============================================================
# Verify placement
# ============================================================

gc.collect()
torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
free_bytes, total_bytes = torch.cuda.mem_get_info()

print("\n" + "=" * 80)
print("MODEL LOAD SUCCESS")
print("=" * 80)

print(f"Model:             {BASE_MODEL}")
print("Quantization:      NF4 4-bit")
print("Compute dtype:     FP16")
print("Device:            CUDA:0")
print(f"GPU allocated:     {allocated:.2f} GB")
print(f"GPU reserved:      {reserved:.2f} GB")
print(f"GPU free:          {free_bytes / 1024**3:.2f} GB")
print(f"GPU total:         {total_bytes / 1024**3:.2f} GB")

print("=" * 80)
print("✅ Base model loaded entirely on the T4.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


QWEN3 4B NF4 MODEL LOAD
GPU: Tesla T4
GPU free before model load: 14.42 GB / 14.56 GB

Loading Qwen/Qwen3-4B-Instruct-2507 in 4-bit NF4 (FP16 compute) onto Tesla T4...



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


MODEL LOAD SUCCESS
Model:             Qwen/Qwen3-4B-Instruct-2507
Quantization:      NF4 4-bit
Compute dtype:     FP16
Device:            CUDA:0
GPU allocated:     2.49 GB
GPU reserved:      2.53 GB
GPU free:          11.87 GB
GPU total:         14.56 GB
✅ Base model loaded entirely on the T4.


In [ ]:
# CELL 11: Attach LoRA Adapter (r=16, alpha=32)
from src.models.qwen4b.lora import apply_qlora, get_qlora_config

lora_config = get_qlora_config(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = apply_qlora(model, lora_config=lora_config, prepare_kbit=True)
print("LoRA adapter attached successfully!")

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
LoRA adapter attached successfully!


In [ ]:
# CELL 12: Inspect Model Parameters
from src.models.qwen4b.loader import get_model_parameters_info
from src.training.qwen4b.utils import print_gpu_environment

print_gpu_environment()
param_info = get_model_parameters_info(model)
print(f"Total Parameters:     {param_info['total_parameters']:,}")
print(f"Trainable Parameters: {param_info['trainable_parameters']:,}")
print(f"Trainable Ratio:      {param_info['trainable_percentage']:.4f}%")


HARDWARE & DEPENDENCY ENVIRONMENT
GPU:                  Tesla T4
VRAM:                 14.56 GB
CUDA Available:       True (Version: 13.0)
PyTorch Version:      2.11.0+cu130
Transformers Version: 5.20.1
PEFT Version:         0.21.2
BitsAndBytes Version: 0.50.2
Python Version:       3.13.15

Total Parameters:     2,238,840,320
Trainable Parameters: 33,030,144
Trainable Ratio:      1.4753%


In [ ]:
# CELL 13: Production-Length Forward / Backward Smoke Test
# Verifies that the REAL training sequence length fits in T4 VRAM.

import gc
import torch

from src.training.qwen4b.utils import (
    run_smoke_forward_backward_step,
)

print("=" * 90)
print("T4 PRODUCTION-LENGTH QLoRA SMOKE TEST")
print("=" * 90)

print(f"Testing sequence length: {MAX_SEQ_LENGTH}")
print("Micro batch size:       1")
print("Gradient checkpointing: ENABLED")
print("Precision:              FP16")
print("KV cache:               DISABLED")

# Make sure model is configured for training
model.train()

if hasattr(model, "config"):
    model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        }
    )

# Clear leftover benchmark/inference memory
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    free_before, total = torch.cuda.mem_get_info()

    print(
        f"\nGPU free before smoke test: "
        f"{free_before / 1024**3:.2f} GB"
    )

    print(
        f"GPU total VRAM:            "
        f"{total / 1024**3:.2f} GB"
    )

print("\nRunning one real forward + backward pass...\n")

success, msg = run_smoke_forward_backward_step(
    model,
    tokenizer,
    max_seq_len=MAX_SEQ_LENGTH,
)

print("\n" + "=" * 90)

if not success:
    print("❌ PRODUCTION-LENGTH SMOKE TEST FAILED")
    print("=" * 90)

    print("\nReason:")
    print(msg)

    print(
        "\nDo NOT start full training yet.\n"
        "Reduce MAX_SEQ_LENGTH and rerun this cell."
    )

    raise RuntimeError(
        f"Production smoke test failed at "
        f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}: {msg}"
    )

print("✅ PRODUCTION-LENGTH SMOKE TEST PASSED")
print("=" * 90)

print(msg)

if torch.cuda.is_available():

    free_after, total = torch.cuda.mem_get_info()

    allocated = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved() / 1024**3
    )

    print(
        f"\nGPU allocated after test: "
        f"{allocated:.2f} GB"
    )

    print(
        f"GPU reserved after test:  "
        f"{reserved:.2f} GB"
    )

    print(
        f"GPU free after test:      "
        f"{free_after / 1024**3:.2f} GB"
    )

print(
    "\nIf this passed at "
    f"{MAX_SEQ_LENGTH} tokens, the full training cell "
    "is much more likely to fit."
)

T4 PRODUCTION-LENGTH QLoRA SMOKE TEST
Testing sequence length: 1024
Micro batch size:       1
Gradient checkpointing: ENABLED
Precision:              FP16
KV cache:               DISABLED

GPU free before smoke test: 10.30 GB
GPU total VRAM:            14.56 GB

Running one real forward + backward pass...


✅ PRODUCTION-LENGTH SMOKE TEST PASSED
Smoke step succeeded with initial loss: 5.9961

GPU allocated after test: 3.36 GB
GPU reserved after test:  4.11 GB
GPU free after test:      10.28 GB

If this passed at 1024 tokens, the full training cell is much more likely to fit.


In [ ]:
# CELL 14: Baseline Benchmarks + Coding Benchmark Sanity Check

# ============================================================
# 0. Ensure correct Project XYZ branch is loaded in Colab
# ============================================================

import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO = Path("/content/project-xyz")
BRANCH = "qwen3-4b-stream-coding-v2"

if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "-b",
            BRANCH,
            "https://github.com/sajancodes/project-xyz.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )

    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "reset",
            "--hard",
            f"origin/{BRANCH}",
        ],
        check=True,
    )

os.chdir(REPO)

repo_path = str(REPO)

if repo_path in sys.path:
    sys.path.remove(repo_path)

sys.path.insert(0, repo_path)

# Clear stale imports from previous branch/runtime state
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

importlib.invalidate_caches()

print("=" * 100)
print("PROJECT ENVIRONMENT")
print("=" * 100)
print("Working directory:", os.getcwd())
print("Repository:", REPO)
print("Branch:", BRANCH)

assert (REPO / "src").exists(), "src/ directory missing."
assert (
    REPO / "src/evaluation/coding/runner.py"
).exists(), "Coding benchmark runner missing."

assert "model" in globals(), (
    "Model missing from memory. "
    "Rerun the model-loading and LoRA attachment cells."
)

assert "tokenizer" in globals(), (
    "Tokenizer missing from memory. "
    "Rerun the tokenizer-loading cell."
)

print("✅ Repository/import environment ready.\n")


# ============================================================
# 1. Imports
# ============================================================

from src.evaluation.coding.runner import (
    clean_code_completion,
    evaluate_coding_benchmark,
    execute_code_sandbox,
)

from src.evaluation.coding.tasks import CODING_BENCHMARK_TASKS

from src.evaluation.qwen4b.benchmark import run_benchmark

from src.models.qwen4b.inference import generate_response


# ============================================================
# 2. Put model into fast inference mode
# ============================================================

model.eval()

try:
    model.gradient_checkpointing_disable()
    print("✅ Gradient checkpointing disabled for benchmark inference.")
except Exception as e:
    print("Gradient checkpointing disable skipped:", e)

try:
    model.config.use_cache = True
    print("✅ KV cache enabled for benchmark generation.")
except Exception as e:
    print("KV cache enable skipped:", e)


# ============================================================
# 3. Multi-domain baseline benchmark
# ============================================================

print("\n" + "=" * 100)
print("1. BASELINE MULTI-DOMAIN BENCHMARK")
print("=" * 100)

base_multi_results = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    tag="base_v2",
    output_dir=f"{DRIVE_ROOT}/benchmarks",
)

print(
    "\nBaseline Multi-Domain Score: "
    f"{base_multi_results['overall_score']:.4f}"
)

print("\nCategory Scores:")

for category, score in base_multi_results["category_scores"].items():
    print(f"  {category:<35} {score:.4f}")


# ============================================================
# 4. Coding benchmark sanity check
# ============================================================

print("\n" + "=" * 100)
print("2. CODING BENCHMARK SANITY CHECK")
print("=" * 100)

print(
    "\nBefore running all 60 tasks, "
    "the first 3 coding tasks will be inspected in detail."
)

probe_results = []


for index, task in enumerate(
    CODING_BENCHMARK_TASKS[:3],
    start=1,
):

    print("\n\n" + "#" * 100)

    print(
        f"CODING PROBE {index}/3 — "
        f"{task.task_id} ({task.entry_point})"
    )

    print("#" * 100)

    print("\nTASK PROMPT:")
    print(task.prompt)

    prompt_text = (
        "Complete the following Python function.\n"
        "Return ONLY valid Python code.\n"
        "Do not explain the solution.\n"
        "Do not include Markdown outside the code.\n"
        "Do not repeat the problem statement.\n\n"
        f"```python\n{task.prompt}\n```"
    )

    print("\nGenerating model response...\n")

    raw_response = generate_response(
        model=model,
        tokenizer=tokenizer,
        messages_or_prompt=prompt_text,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
    )


    # --------------------------------------------------------
    # Raw model response
    # --------------------------------------------------------

    print("-" * 100)
    print("RAW MODEL RESPONSE")
    print("-" * 100)

    print(raw_response)


    # --------------------------------------------------------
    # Extract executable Python
    # --------------------------------------------------------

    extracted_code = clean_code_completion(
        raw_response,
        task.entry_point,
    )

    # If the model returned only a function body,
    # prepend the provided signature/stub.
    if f"def {task.entry_point}" not in extracted_code:
        extracted_code = (
            task.prompt
            + "\n"
            + extracted_code
        )


    print("\n" + "-" * 100)
    print("CODE SENT TO PYTHON TEST RUNNER")
    print("-" * 100)

    print(extracted_code)


    # --------------------------------------------------------
    # Show hidden unit tests
    # --------------------------------------------------------

    print("\n" + "-" * 100)
    print("UNIT TESTS")
    print("-" * 100)

    print(task.test_cases)


    # --------------------------------------------------------
    # Execute candidate code
    # --------------------------------------------------------

    passed, error_message = execute_code_sandbox(
        extracted_code,
        task.test_cases,
        timeout=5.0,
    )

    probe_results.append(passed)


    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    print("\n" + "-" * 100)

    if passed:
        print("✅ RESULT: PASS")

    else:
        print("❌ RESULT: FAIL")

        print("\nEXACT ERROR:")
        print(error_message)

    print("-" * 100)


# ============================================================
# 5. Coding probe summary
# ============================================================

probe_passes = sum(probe_results)

print("\n\n" + "=" * 100)
print("CODING SANITY CHECK SUMMARY")
print("=" * 100)

print(f"Probe tasks passed: {probe_passes}/3")


# ============================================================
# 6. Decide whether to run full 60-task benchmark
# ============================================================

if probe_passes == 0:

    print(
        """
⚠️ FULL 60-TASK BENCHMARK STOPPED.

All 3 basic coding probes failed.

This is suspicious enough that the generated code and exact
errors should be inspected before trusting a 0/60 result.

The expensive 60-task benchmark has NOT been started.
"""
    )

    base_coding_results = None


else:

    print(
        """
✅ At least one sanity-check task passed.

The benchmark execution pipeline appears capable of accepting
model-generated Python.

Proceeding with the complete 60-task execution benchmark...
"""
    )

    print("\n" + "=" * 100)
    print("3. FULL 60-TASK EXECUTION BENCHMARK")
    print("=" * 100)


    base_coding_results = evaluate_coding_benchmark(
        model=model,
        tokenizer=tokenizer,
        output_path=(
            f"{DRIVE_ROOT}/benchmarks/"
            "base_coding_benchmark.json"
        ),
    )


    # --------------------------------------------------------
    # Full benchmark result
    # --------------------------------------------------------

    print("\n" + "=" * 100)
    print("BASELINE CODING RESULT")
    print("=" * 100)

    print(
        f"Pass@1: "
        f"{base_coding_results['pass_at_1']}%"
    )

    print(
        f"Passed: "
        f"{base_coding_results['passed_count']}/"
        f"{base_coding_results['num_tasks']}"
    )

    print(
        "\nSaved to:\n"
        f"{DRIVE_ROOT}/benchmarks/"
        "base_coding_benchmark.json"
    )


print("\n" + "=" * 100)
print("CELL 14 COMPLETE")
print("=" * 100)

PROJECT ENVIRONMENT
Working directory: /content/project-xyz
Repository: /content/project-xyz
Branch: qwen3-4b-stream-coding-v2
✅ Repository/import environment ready.

✅ Gradient checkpointing disabled for benchmark inference.
✅ KV cache enabled for benchmark generation.

1. BASELINE MULTI-DOMAIN BENCHMARK

Baseline Multi-Domain Score: 0.6468

Category Scores:
  networking_security                 0.8750
  secure_coding                       0.3125
  vulnerability_analysis              0.5268
  incident_response                   0.5000
  log_analysis                        0.6667
  web_api_security                    1.0000
  cloud_container_security            0.7500
  general_instruction                 0.5000
  general_knowledge                   0.4000
  coding                              0.7500
  reasoning                           0.8750

2. CODING BENCHMARK SANITY CHECK

Before running all 60 tasks, the first 3 coding tasks will be inspected in detail.


#######################

In [ ]:
# CELL 15: Initialize 5K-Step Streaming Trainer and Progressive Datasets

from pathlib import Path

import gc
import torch

from transformers import (
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from src.streaming.streamer import StreamSFTDataset
from src.training.qwen4b.callbacks import LiveTrainingProgressCallback
from src.training.qwen4b.checkpointing import (
    DriveBackupCallback,
    GoogleDriveCheckpointManager,
)
from src.training.qwen4b.collator import DataCollatorForAssistantOnlyLM
from src.training.qwen4b.dataset import CyberChatDataset


# ============================================================
# 0. 5K TRAINING RUN CONFIGURATION
# ============================================================

# Override the earlier notebook setting.
MAX_STEPS = 5000

# T4-safe context length.
# Make sure Cell 7 also uses 1024 so every other cell agrees.
MAX_SEQ_LENGTH = 1024

# Effective batch:
# 1 microbatch × 16 gradient accumulation = 16
EFFECTIVE_BATCH_SIZE = (
    MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION
)

# Approximate number of individual examples consumed
# across 5,000 optimizer steps.
STREAM_TARGET = (
    MAX_STEPS * EFFECTIVE_BATCH_SIZE
)

print("=" * 90)
print("5K STREAMING TRAINING RUN")
print("=" * 90)

print(f"Optimizer steps:          {MAX_STEPS:,}")
print(f"Micro batch size:         {MICRO_BATCH_SIZE}")
print(f"Gradient accumulation:    {GRADIENT_ACCUMULATION}")
print(f"Effective batch size:     {EFFECTIVE_BATCH_SIZE}")
print(f"Target example exposure:  {STREAM_TARGET:,}")
print(f"Max sequence length:      {MAX_SEQ_LENGTH}")

print("=" * 90)


# ============================================================
# 1. Restore model to TRAINING mode
# ============================================================

model.train()

if hasattr(model, "config"):
    model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        }
    )

print("✅ Model restored to training configuration.")


# ============================================================
# 2. Clear residual inference / benchmark CUDA memory
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    allocated_gb = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved_gb = (
        torch.cuda.memory_reserved() / 1024**3
    )

    free_bytes, total_bytes = torch.cuda.mem_get_info()

    free_gb = free_bytes / 1024**3
    total_gb = total_bytes / 1024**3

    print("\nGPU memory before Trainer initialization:")
    print(f"  Allocated: {allocated_gb:.2f} GB")
    print(f"  Reserved:  {reserved_gb:.2f} GB")
    print(f"  Free:      {free_gb:.2f} GB")
    print(f"  Total:     {total_gb:.2f} GB")


# ============================================================
# 3. Google Drive checkpoint manager
# ============================================================

drive_mgr = GoogleDriveCheckpointManager(
    DRIVE_ROOT
)

drive_mgr.ensure_directories()

print(f"\n✅ Drive root: {DRIVE_ROOT}")


# ============================================================
# 4. Streaming source quotas for ~80K examples
# ============================================================

# Same mixture:
#
# 45% OpenCoder
# 25% BigCode
# 20% SmolTalk
# 10% Cybersecurity
#
# 80,000 total target:
#
# OpenCoder     36,000
# BigCode       20,000
# SmolTalk      16,000
# Cybersecurity  8,000

STREAM_QUOTAS = {
    "opencoder": 36000,
    "bigcode": 20000,
    "smoltalk": 16000,
    "cybersecurity": 8000,
}

assert sum(STREAM_QUOTAS.values()) == STREAM_TARGET, (
    f"Quota total {sum(STREAM_QUOTAS.values())} "
    f"does not match STREAM_TARGET {STREAM_TARGET}"
)

print("\nStreaming quotas:")

for source, target in STREAM_QUOTAS.items():
    print(
        f"  {source:<16} "
        f"{target:>8,}"
    )


# ============================================================
# 5. Streaming training dataset
# ============================================================

train_stream = StreamSFTDataset(
    tokenizer=tokenizer,

    partition="train",

    max_seq_length=MAX_SEQ_LENGTH,

    total_target=STREAM_TARGET,

    quotas=STREAM_QUOTAS,

    shuffle_buffer_size=10000,

    seed=42,
)

print(
    "\n✅ Streaming training dataset initialized "
    f"for up to {STREAM_TARGET:,} examples."
)


# ============================================================
# 6. Finite validation dataset
# ============================================================

val_dataset = CyberChatDataset(
    val_path,

    tokenizer=tokenizer,

    max_length=MAX_SEQ_LENGTH,
)

print(
    f"✅ Validation examples: "
    f"{len(val_dataset):,}"
)


# ============================================================
# 7. Assistant-only loss collator
# ============================================================

collator = DataCollatorForAssistantOnlyLM(
    tokenizer=tokenizer
)

print("✅ Assistant-only loss collator initialized.")


# ============================================================
# 8. TrainingArguments
# ============================================================

training_args = TrainingArguments(

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    output_dir=(
        "outputs/"
        "qwen3_4b_coding_stream_v2_5k"
    ),


    # --------------------------------------------------------
    # Batch configuration
    # --------------------------------------------------------

    per_device_train_batch_size=MICRO_BATCH_SIZE,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION
    ),


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    learning_rate=LEARNING_RATE,

    weight_decay=0.01,

    # Current Transformers API.
    # 0.05 = approximately 5% warmup.
    warmup_steps=WARMUP_RATIO,

    lr_scheduler_type="cosine",


    # --------------------------------------------------------
    # Streaming training length
    # --------------------------------------------------------

    max_steps=MAX_STEPS,


    # --------------------------------------------------------
    # Memory optimization for T4
    # --------------------------------------------------------

    gradient_checkpointing=True,

    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=True,

    bf16=False,

    optim="paged_adamw_8bit",


    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=10,

    logging_first_step=True,


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    eval_strategy="steps",

    # 50 evaluations across 5K steps
    eval_steps=100,


    # --------------------------------------------------------
    # Checkpointing
    # --------------------------------------------------------

    save_strategy="steps",

    # Saves every 100 optimizer steps.
    save_steps=100,

    save_total_limit=3,


    # --------------------------------------------------------
    # Best model
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=42,

    data_seed=42,


    # --------------------------------------------------------
    # Misc
    # --------------------------------------------------------

    report_to="none",

    remove_unused_columns=False,
)


# ============================================================
# 9. Callbacks
# ============================================================

callbacks = [

    DriveBackupCallback(
        drive_mgr
    ),

    LiveTrainingProgressCallback(
        max_steps=MAX_STEPS,

        log_every_n_steps=10,

        tracker=train_stream.tracker,
    ),

    EarlyStoppingCallback(
        early_stopping_patience=3,
    ),
]


# ============================================================
# 10. Trainer
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_stream,

    eval_dataset=val_dataset,

    data_collator=collator,

    callbacks=callbacks,
)


# ============================================================
# 11. Resume checkpoint
# ============================================================

LOCAL_OUTPUT_DIR = Path(
    "outputs/"
    "qwen3_4b_coding_stream_v2_5k"
)

resume_target = (
    RESUME_FROM_CHECKPOINT
    or drive_mgr.get_resume_checkpoint(
        LOCAL_OUTPUT_DIR
    )
)


# ============================================================
# 12. Final training summary
# ============================================================

print("\n" + "=" * 90)
print("5K STREAMING TRAINER INITIALIZED")
print("=" * 90)

print(f"Base model:                {BASE_MODEL}")

print(
    f"Max sequence length:       "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Micro batch size:          "
    f"{MICRO_BATCH_SIZE}"
)

print(
    f"Gradient accumulation:     "
    f"{GRADIENT_ACCUMULATION}"
)

print(
    f"Effective batch size:      "
    f"{EFFECTIVE_BATCH_SIZE}"
)

print(
    f"Max optimizer steps:       "
    f"{MAX_STEPS:,}"
)

print(
    f"Approx example exposures:  "
    f"{STREAM_TARGET:,}"
)

print(
    f"Learning rate:             "
    f"{LEARNING_RATE}"
)

print(
    f"Warmup:                    "
    f"{WARMUP_RATIO * 100:.1f}%"
)

print("Optimizer:                 paged_adamw_8bit")

print("Scheduler:                 cosine")

print("Precision:                 FP16")

print("Gradient checkpointing:    ENABLED")

print("KV cache:                   DISABLED")


print("\nLoRA:")

print(
    f"  r:                        "
    f"{LORA_R}"
)

print(
    f"  alpha:                    "
    f"{LORA_ALPHA}"
)

print(
    f"  dropout:                  "
    f"{LORA_DROPOUT}"
)


print("\nStreaming mixture:")

print("  OpenCoder:                36,000  (45%)")

print("  BigCode:                  20,000  (25%)")

print("  SmolTalk:                 16,000  (20%)")

print("  Cybersecurity:             8,000  (10%)")


print(
    f"\nValidation examples:       "
    f"{len(val_dataset):,}"
)

print(
    f"Resume checkpoint target:  "
    f"{resume_target}"
)


if torch.cuda.is_available():

    free_bytes, total_bytes = torch.cuda.mem_get_info()

    print(
        f"\nGPU free before training:  "
        f"{free_bytes / 1024**3:.2f} GB"
    )

    print(
        f"GPU total:                 "
        f"{total_bytes / 1024**3:.2f} GB"
    )


print("=" * 90)

print(
    "\n✅ 5K-step streaming Trainer initialized.\n"
    "Next run the production-length smoke test at "
    "MAX_SEQ_LENGTH before starting the final training cell."
)

5K STREAMING TRAINING RUN
Optimizer steps:          5,000
Micro batch size:         1
Gradient accumulation:    16
Effective batch size:     16
Target example exposure:  80,000
Max sequence length:      1024
✅ Model restored to training configuration.

GPU memory before Trainer initialization:
  Allocated: 3.36 GB
  Reserved:  4.11 GB
  Free:      10.28 GB
  Total:     14.56 GB

✅ Drive root: /content/drive/MyDrive/ProjectXYZ/qwen3-4b-coding-stream-v2

Streaming quotas:
  opencoder          36,000
  bigcode            20,000
  smoltalk           16,000
  cybersecurity       8,000

✅ Streaming training dataset initialized for up to 80,000 examples.
✅ Validation examples: 600
✅ Assistant-only loss collator initialized.

5K STREAMING TRAINER INITIALIZED
Base model:                Qwen/Qwen3-4B-Instruct-2507
Max sequence length:       1024
Micro batch size:          1
Gradient accumulation:     16
Effective batch size:      16
Max optimizer steps:       5,000
Approx example exposures:  80,

In [ ]:
# CELL 16: Train + Maintain ONE Rolling Checkpoint

from pathlib import Path
import shutil
import os
import torch
from transformers import TrainerCallback

# ============================================================
# CONFIG
# ============================================================

SAVE_EVERY = 100

ROLLING_CHECKPOINT = Path(
    "/content/drive/MyDrive/ProjectXYZ/"
    "qwen3-4b-coding-stream-v2/latest_checkpoint"
)

ROLLING_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

print("Rolling checkpoint:")
print(ROLLING_CHECKPOINT)


# ============================================================
# CALLBACK
# ============================================================

class RollingCheckpointCallback(TrainerCallback):

    def on_save(self, args, state, control, **kwargs):

        step = state.global_step

        source = Path(args.output_dir) / f"checkpoint-{step}"

        if not source.exists():
            print(f"\n⚠️ Could not find checkpoint source: {source}")
            return control

        print("\n" + "=" * 80)
        print(f"💾 Updating rolling checkpoint at step {step}")
        print("=" * 80)

        # ----------------------------------------------------
        # Copy to temporary directory first
        # so latest_checkpoint is never half-written
        # ----------------------------------------------------

        temp_checkpoint = Path(
            str(ROLLING_CHECKPOINT) + "_tmp"
        )

        if temp_checkpoint.exists():
            shutil.rmtree(temp_checkpoint)

        shutil.copytree(
            source,
            temp_checkpoint
        )

        # ----------------------------------------------------
        # Replace previous rolling checkpoint
        # ----------------------------------------------------

        if ROLLING_CHECKPOINT.exists():
            shutil.rmtree(ROLLING_CHECKPOINT)

        temp_checkpoint.rename(
            ROLLING_CHECKPOINT
        )

        print(
            f"✅ Step {step} permanently saved to:\n"
            f"{ROLLING_CHECKPOINT}"
        )

        # ----------------------------------------------------
        # Delete numbered checkpoints
        # AFTER rolling copy succeeded
        # ----------------------------------------------------

        output_path = Path(args.output_dir)

        for checkpoint in output_path.glob("checkpoint-*"):

            try:
                shutil.rmtree(checkpoint)

                print(
                    f"🗑 Removed temporary checkpoint: "
                    f"{checkpoint.name}"
                )

            except Exception as e:

                print(
                    f"⚠️ Could not remove "
                    f"{checkpoint.name}: {e}"
                )

        print("=" * 80)

        return control


# ============================================================
# ADD CALLBACK ONLY ONCE
# ============================================================

trainer.remove_callback(RollingCheckpointCallback)

trainer.add_callback(
    RollingCheckpointCallback()
)


# ============================================================
# RESUME AUTOMATICALLY IF CHECKPOINT EXISTS
# ============================================================

resume_checkpoint = None

if ROLLING_CHECKPOINT.exists():

    trainer_state = (
        ROLLING_CHECKPOINT /
        "trainer_state.json"
    )

    if trainer_state.exists():

        resume_checkpoint = str(
            ROLLING_CHECKPOINT
        )

        print("\n🔄 Existing checkpoint found.")
        print("Resuming from:")
        print(resume_checkpoint)

    else:

        print(
            "\n⚠️ Rolling checkpoint exists "
            "but trainer_state.json is missing."
        )

        print("Starting training normally.")

else:

    print("\n🆕 No previous checkpoint found.")
    print("Starting new training.")


# ============================================================
# GPU STATUS
# ============================================================

if torch.cuda.is_available():

    free, total = torch.cuda.mem_get_info()

    print(
        f"\nGPU: {torch.cuda.get_device_name(0)}"
    )

    print(
        f"Free VRAM: "
        f"{free / 1024**3:.2f} GB / "
        f"{total / 1024**3:.2f} GB"
    )


# ============================================================
# START TRAINING
# ============================================================

print("\n" + "=" * 80)
print("STARTING STREAMING QLORA TRAINING")
print("=" * 80)

try:

    train_output = trainer.train(
        resume_from_checkpoint=resume_checkpoint
    )

    print("\n✅ TRAINING COMPLETED")

except KeyboardInterrupt:

    print("\n⚠️ Training manually interrupted.")

    print(
        "Your most recently completed rolling checkpoint "
        "is still safely stored in Google Drive."
    )

except Exception as e:

    print("\n❌ TRAINING STOPPED WITH ERROR")

    print(type(e).__name__)
    print(e)

    print(
        "\nThe latest successfully completed checkpoint "
        "should still be available at:"
    )

    print(ROLLING_CHECKPOINT)

    raise


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 80)

if ROLLING_CHECKPOINT.exists():

    print("✅ Latest resumable checkpoint:")

    print(
        ROLLING_CHECKPOINT
    )

else:

    print(
        "⚠️ No rolling checkpoint has been "
        "created yet."
    )

print("=" * 80)

Starting Experiment V2 Streaming Training on NVIDIA T4...

STREAMING QLORA TRAINING INITIALIZED
Target Max Steps: 5000 | Effective Batch Size: 16



Step,Training Loss,Validation Loss
100,0.983975,0.601150


Step [>--------------------]   0.2% (  10/5000) | Loss: 2.0460 | LR: 0.00e+00 | Time: 00:03:52 | ETA: 08:09:50 | VRAM: 3.4G/11.4G
Step [>--------------------]   0.4% (  20/5000) | Loss: 1.8241 | LR: 1.80e-06 | Time: 00:07:28 | ETA: 07:00:49 | VRAM: 3.4G/11.4G
Step [>--------------------]   0.6% (  30/5000) | Loss: 1.8317 | LR: 3.80e-06 | Time: 00:11:05 | ETA: 06:36:48 | VRAM: 3.4G/11.4G
Step [>--------------------]   0.8% (  40/5000) | Loss: 1.7707 | LR: 5.80e-06 | Time: 00:14:38 | ETA: 06:15:26 | VRAM: 3.4G/11.4G
Step [>--------------------]   1.0% (  50/5000) | Loss: 1.5799 | LR: 7.80e-06 | Time: 00:18:12 | ETA: 06:02:02 | VRAM: 3.4G/11.4G
Step [>--------------------]   1.2% (  60/5000) | Loss: 1.3951 | LR: 9.80e-06 | Time: 00:21:48 | ETA: 05:54:52 | VRAM: 3.4G/11.4G
Step [>--------------------]   1.4% (  70/5000) | Loss: 1.1582 | LR: 1.18e-05 | Time: 00:25:15 | ETA: 05:38:59 | VRAM: 3.4G/11.4G
Step [>--------------------]   1.6% (  80/5000) | Loss: 1.0532 | LR: 1.38e-05 | Time: 00:2

In [ ]:
# CELL 17: Save Final LoRA Adapter Locally + Google Drive

from pathlib import Path
import shutil

local_adapter = Path(
    "/content/project-xyz/outputs/qwen3_4b_coding_stream_v2/final_adapter"
)

drive_adapter = Path(
    "/content/drive/MyDrive/ProjectXYZ/"
    "qwen3-4b-coding-stream-v2/final_adapter"
)

local_adapter.mkdir(parents=True, exist_ok=True)
drive_adapter.parent.mkdir(parents=True, exist_ok=True)

# Make sure training objects still exist
if "trainer" not in globals():
    raise RuntimeError(
        "trainer is not available. You are probably in a new/restarted kernel."
    )

if trainer.model is None:
    raise RuntimeError("trainer.model is None.")

if "tokenizer" not in globals():
    raise RuntimeError("tokenizer is not available.")

# Save LoRA adapter
trainer.model.save_pretrained(str(local_adapter))
tokenizer.save_pretrained(str(local_adapter))

print("✅ Saved locally:")
print(local_adapter)

# Copy entire adapter directory to Drive
if drive_adapter.exists():
    shutil.rmtree(drive_adapter)

shutil.copytree(local_adapter, drive_adapter)

print("\n✅ Saved permanently to Google Drive:")
print(drive_adapter)

NameError: name 'Path' is not defined

In [ ]:
# CELL 18: Clean Training State from GPU Memory
from src.training.qwen4b.utils import clear_cuda_memory

del trainer
del model
clear_cuda_memory()
print("Training state cleared from GPU memory.")

In [ ]:
# CELL 19: Reload Best Adapter from Google Drive
from src.models.qwen4b.inference import load_lora_adapter
from src.models.qwen4b.loader import load_qwen_model

print("Reloading base model in NF4 4-bit...")
eval_base_model = load_qwen_model(BASE_MODEL, load_in_4bit=True, device_map="auto")

adapter_path = f"{DRIVE_ROOT}/adapters/final_stream_coding_adapter"
print(f"Attaching fine-tuned streaming adapter: {adapter_path}...")
eval_model = load_lora_adapter(eval_base_model, adapter_path)
print("Adapter reloaded successfully!")

In [ ]:
# CELL 20: Post-Training Benchmarks
from src.evaluation.coding.runner import evaluate_coding_benchmark
from src.evaluation.qwen4b.benchmark import run_benchmark

print("1. Running Post-Training Multi-Domain Benchmark...")
post_multi_results = run_benchmark(
    model=eval_model,
    tokenizer=tokenizer,
    tag="qlora_v2_stream",
    output_dir=f"{DRIVE_ROOT}/benchmarks",
)
print(f"Post-Training Multi-Domain Score: {post_multi_results['overall_score']:.4f}")

print("\n2. Running Post-Training Execution-Based Coding Benchmark (60 Tasks)...\n")
post_coding_results = evaluate_coding_benchmark(
    model=eval_model,
    tokenizer=tokenizer,
    output_path=f"{DRIVE_ROOT}/benchmarks/post_coding_benchmark.json",
)
print(f"Post-Training Coding Pass@1: {post_coding_results['pass_at_1']}% ({post_coding_results['passed_count']}/{post_coding_results['num_tasks']})")

In [ ]:
# CELL 21: Generate Before vs After Comparison Report
from IPython.display import Markdown, display
from src.evaluation.qwen4b.comparison import compare_benchmarks

report_md = compare_benchmarks(
    base_results=base_multi_results,
    qlora_results=post_multi_results,
    output_md_path=f"{DRIVE_ROOT}/reports/comparison_report_v2.md",
)

coding_summary = (
    f"\n### Execution-Based Coding Benchmark (60 Tasks)\n\n"
    f"| Model | Tasks Passed | Total Tasks | Pass@1 Accuracy |\n"
    f"|---|---:|---:|---:|\n"
    f"| Base Qwen 4B | {base_coding_results['passed_count']} | {base_coding_results['num_tasks']} | {base_coding_results['pass_at_1']}% |\n"
    f"| Streaming QLoRA v2 | {post_coding_results['passed_count']} | {post_coding_results['num_tasks']} | {post_coding_results['pass_at_1']}% |\n"
    f"| **Delta** | **+{post_coding_results['passed_count'] - base_coding_results['passed_count']}** | -- | **+{post_coding_results['pass_at_1'] - base_coding_results['pass_at_1']:.2f}%** |\n"
)

display(Markdown(report_md + coding_summary))

In [ ]:
# CELL 22: Interactive Code & Cybersecurity Query Testing
from src.models.qwen4b.inference import generate_response

prompt = "Write a Python function to find the longest palindromic substring in a string, with full unit tests." # @param {type:"string"}
response = generate_response(eval_model, tokenizer, prompt, max_new_tokens=512)
print("Prompt:", prompt)
print("\nResponse:\n", response)